# MEIDNet on your own data

You need:

1. a **table** (CSV/Excel) with one row per material: an id column, one or more numeric property columns, and either a `cif` column with the CIF text or a folder of `<id>.cif` files;
2. to know which **material family** your structures belong to (built in: cubic ABX₃ perovskites, A₂BB′X₆ double perovskites; or describe your own in a YAML file).

Run the cells in order. Each step writes an HTML report that says what happened in plain language.

In [ ]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install git+https://github.com/ABnano/MEIDNet.git


## 1. Upload your data
Either upload a CSV with a `cif` column, or a CSV plus a zip of CIF files.

In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile, os
for name in uploaded:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('structures')
        print('extracted', len(os.listdir('structures')), 'files to structures/')
table = next(n for n in uploaded if n.endswith(('.csv', '.xlsx')))
print('table:', table)

In [ ]:
import pandas as pd
df = pd.read_csv(table) if table.endswith('.csv') else pd.read_excel(table)
print(df.shape); df.head()

## 2. Describe your data in `meidnet.yaml`
Fill in the column names and the family. `meidnet init` writes a commented file; you can edit it afterwards.

In [ ]:
id_column = 'material_id'                 #@param {type:'string'}
properties = 'band_gap formation_energy'  #@param {type:'string'}
cif_column = 'cif'                        #@param {type:'string'}
family = 'perovskite_abx3'                #@param ['perovskite_abx3', 'double_perovskite_a2bbx6']
variant = 'oxide'                         #@param {type:'string'}
!meidnet init --name mine --table "{table}" --id-column {id_column} --cif-column {cif_column} --properties {properties} --family {family} --variant {variant} --force
import os
if os.path.isdir('structures'):
    import re
    y = open('meidnet.yaml').read().replace('cif_column: cif', 'cif_column: null\n  structures_dir: structures')
    open('meidnet.yaml','w').write(y)
print(open('meidnet.yaml').read())

## 3. Check
Reads every row exactly as training would and reports what is usable and why rows were skipped.

In [ ]:
!meidnet check meidnet.yaml

In [ ]:
from IPython.display import HTML, display
display(HTML(open('runs/mine/check_report.html', encoding='utf-8').read()))

## 4. Train
Start with a few epochs to see the pipeline work, then raise `epochs` (200 is typical). Colab's free GPU is enough: Runtime → Change runtime type → GPU.

In [ ]:
epochs = 20  #@param {type:'integer'}
!meidnet train meidnet.yaml --epochs {epochs}

In [ ]:
from IPython.display import HTML, display
display(HTML(open('runs/mine/training_report.html', encoding='utf-8').read()))

## 5. Set targets and generate
Edit the `targets:` in `meidnet.yaml` (or below) and run.

In [ ]:
import yaml
cfg = yaml.safe_load(open('meidnet.yaml'))
props = [p['column'] for p in cfg['data']['properties']]
print('properties:', props)
# example: first property at its training median; edit freely
cfg['generation']['targets'] = [{p: float(df[p].median()) for p in props}]
cfg['generation']['per_target'] = 3
cfg['generation']['rounds'] = 5
cfg['generation']['steps'] = 300
yaml.safe_dump(cfg, open('meidnet.yaml','w'), sort_keys=False)
print(cfg['generation']['targets'])

In [ ]:
!meidnet generate meidnet.yaml

In [ ]:
from IPython.display import HTML, display
display(HTML(open('runs/mine/generation_report.html', encoding='utf-8').read()))

## 6. Download everything
CIFs, `candidates.csv`, `model.pt`, the three reports and the exact `meidnet.yaml` that produced them.

In [ ]:
!zip -qr meidnet_results.zip runs/mine meidnet.yaml
files.download('meidnet_results.zip')

## Where to go next
* **Rules**: `exclude_elements`, `overrides` and family files — [recipes](https://abnano.github.io/MEIDNet/recipes/elements/).
* **Live exploration**: `meidnet studio meidnet.yaml` on your computer.
* **Stability**: `pip install 'meidnet[stability]'` then `meidnet screen runs/mine/generation`.